# Sea Ice Classification & Freeboard Pegasus Workflow

A Pegasus workflow for scalable, higher resolution polar sea ice classification
and freeboard calculation from ICESat-2 ATL03 photon-level data.

Based on: *"Scalable Higher Resolution Polar Sea Ice Classification and Freeboard
Calculation from ICESat-2 ATL03 Data"* (Iqrah et al., IPDPSW 2025)

The workflow downloads ICESat-2 ATL03 photon data and coincident Sentinel-2 imagery,
preprocesses the photons into 2m along-track segments, auto-labels them using S2
band ratios, trains an LSTM or MLP classifier, runs inference to classify each
segment as thick ice, thin ice, or open water, and calculates freeboard using
sliding-window sea surface detection.

When `MAX_GRANULES` is set, classification is parallelized with one job per granule
(fan-out), merged back (fan-in), then fed into freeboard calculation.

## Containers

All tools required to execute the jobs are included in a single GPU-enabled container on DockerHub:

[Sea Ice ICESat-2 Container](https://hub.docker.com/r/kthare10/seaice-icesat2) defined in `Docker/Seaice_Dockerfile` with:
* numpy, pandas, scipy, scikit-learn, h5py, matplotlib (base analysis)
* tensorflow with CUDA support (GPU training & inference)
* rasterio, pyproj, cartopy (geospatial processing)
* earthaccess, planetary-computer, pystac-client (data download)

## Accessing the Input Data

ATL03 photon data is downloaded from **NASA Earthdata**. An account is required:
register at [https://urs.earthdata.nasa.gov/](https://urs.earthdata.nasa.gov/).
A **bearer token** (recommended) or username/password must be provided.

Sentinel-2 imagery is downloaded from **Microsoft Planetary Computer** (no credentials required).

## Workflow

| Stage | Description | Memory | GPU |
|-------|-------------|--------|-----|
| `download_atl03` | Fetch ICESat-2 ATL03 HDF5 granules from NASA Earthdata | 4 GB | No |
| `download_sentinel2` | Fetch coincident Sentinel-2 imagery | 4 GB | No |
| `preprocess_atl03` | Filter photons, resample to 2m segments, compute features | 8 GB | No |
| `auto_label` | Co-register S2 with ATL03, overlay labels | 4 GB | No |
| `train_model` | Train LSTM or MLP classifier on labeled data | 16 GB | Yes |
| `classify_seaice` | Run inference (parallelized per granule when `MAX_GRANULES` is set) | 8 GB | Yes |
| `merge_classifications` | Concatenate per-granule CSVs (parallel mode only) | 4 GB | No |
| `calculate_freeboard` | Sliding-window sea surface detection + freeboard | 8 GB | No |
| `visualize_results` | Generate maps, profiles, summary statistics | 4 GB | No |

In [ ]:
!pip3 install pandas h5py

## 1. Create the Sea Ice Workflow

First, configure the region, date range, and workflow parameters.

In [ ]:
# --- Workflow Configuration ---

# Region of interest
# Choices: ross_sea, weddell_sea, beaufort_sea, arctic_ocean, southern_ocean
REGION = 'ross_sea'

# Date range for ATL03 / Sentinel-2 data
START_DATE = '2019-11-01'
END_DATE = '2019-11-30'

# Classifier model type: 'lstm' (96.56% accuracy) or 'mlp' (91.80%)
MODEL_TYPE = 'lstm'

# Limit downloads for faster testing (set to None for full download)
MAX_GRANULES = 3      # Max ATL03 granules (None = all)
MAX_SCENES = 5        # Max Sentinel-2 scenes (None = all)

# Set to True to use synthetic test data (no downloads, no credentials needed)
TEST_MODE = False

### NASA Earthdata Credentials

A bearer token is recommended (especially on FABRIC where `urs.earthdata.nasa.gov` may be unreachable).

1. Register at [https://urs.earthdata.nasa.gov/](https://urs.earthdata.nasa.gov/)
2. Generate a token at `https://urs.earthdata.nasa.gov/users/<username>/user_tokens`
3. Replace `YOUR_TOKEN_HERE` below

In [ ]:
import os

# Option 1: Bearer token (recommended for FABRIC)
os.environ['EARTHDATA_TOKEN'] = 'YOUR_TOKEN_HERE'

# Option 2: Username/password (uncomment and fill in)
# os.environ['EARTHDATA_USERNAME'] = 'your_username'
# os.environ['EARTHDATA_PASSWORD'] = 'your_password'

**Test Mode:** To run end-to-end without downloading real data, set `TEST_MODE = True` above and run the cell below to generate synthetic data. No Earthdata credentials are needed in test mode.

In [ ]:
if TEST_MODE:
    !python generate_test_data.py
    print('Synthetic test data generated.')
else:
    print('Using real data mode - Earthdata credentials required.')

In [ ]:
import os
import sys
import logging
from pathlib import Path
from datetime import datetime, timedelta

from Pegasus.api import *

logging.basicConfig(level=logging.INFO)


class SeaIceWorkflow:
    """Sea ice classification and freeboard workflow for ACCESS/FABRIC."""

    wf = None
    sc = None
    tc = None
    rc = None
    props = None

    dagfile = None
    wf_dir = None
    shared_scratch_dir = None
    local_storage_dir = None
    wf_name = 'seaice'

    def __init__(self, region, start_date, end_date, model_type='lstm',
                 max_granules=None, max_scenes=None, test_mode=False,
                 dagfile='workflow.yml'):
        self.dagfile = dagfile
        self.wf_dir = str(Path('.').resolve())
        self.shared_scratch_dir = os.path.join(self.wf_dir, 'scratch')
        self.local_storage_dir = os.path.join(self.wf_dir, 'output')
        self.region = region
        self.start_date = start_date
        self.end_date = end_date
        self.model_type = model_type
        self.max_granules = max_granules
        self.max_scenes = max_scenes
        self.test_mode = test_mode
        self.earthdata_token = os.environ.get('EARTHDATA_TOKEN')
        self.earthdata_username = os.environ.get('EARTHDATA_USERNAME')
        self.earthdata_password = os.environ.get('EARTHDATA_PASSWORD')

    # --- Write catalogs and DAG ---
    def write(self):
        if self.sc is not None:
            self.sc.write()
        self.props.write()
        self.rc.write()
        self.tc.write()
        try:
            self.wf.write(file=self.dagfile)
        except PegasusClientError as e:
            print(e)

    # --- Plan and submit ---
    def plan_submit(self):
        try:
            self.wf.plan(submit=True)
        except PegasusClientError as e:
            print(e)

    # --- Status ---
    def status(self):
        try:
            self.wf.status(long=True)
        except PegasusClientError as e:
            print(e)

    # --- Wait for completion ---
    def wait(self):
        try:
            self.wf.wait()
        except PegasusClientError as e:
            print(e)

    # --- Statistics ---
    def statistics(self):
        try:
            self.wf.statistics()
        except PegasusClientError as e:
            print(e)

    # --- Pegasus Properties ---
    def create_pegasus_properties(self):
        self.props = Properties()
        self.props['pegasus.transfer.threads'] = '16'
        self.props['pegasus.data.configuration'] = 'condorio'

    # --- Site Catalog ---
    def create_sites_catalog(self, exec_site_name='condorpool'):
        self.sc = SiteCatalog()

        local = (Site('local')
            .add_directories(
                Directory(Directory.SHARED_SCRATCH, self.shared_scratch_dir)
                    .add_file_servers(FileServer('file://' + self.shared_scratch_dir, Operation.ALL)),
                Directory(Directory.LOCAL_STORAGE, self.local_storage_dir)
                    .add_file_servers(FileServer('file://' + self.local_storage_dir, Operation.ALL))
            )
        )

        exec_site = (Site(exec_site_name)
            .add_condor_profile(universe='vanilla')
            .add_pegasus_profile(style='condor')
            .add_pegasus_profile(data_configuration='condorio')
        )

        self.sc.add_sites(local, exec_site)

    # --- Transformation Catalog ---
    def create_transformation_catalog(self, exec_site_name='condorpool'):
        self.tc = TransformationCatalog()

        seaice_container = Container('seaice_container',
            container_type=Container.SINGULARITY,
            image='docker://kthare10/seaice-icesat2:latest',
            image_site='docker_hub',
        )

        seaice_gpu_container = Container('seaice_gpu_container',
            container_type=Container.SINGULARITY,
            image='docker://kthare10/seaice-icesat2:latest',
            image_site='docker_hub',
        ).add_env(SINGULARITY_ARGS='--nv')

        download_atl03 = Transformation('download_atl03', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/download_atl03.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='4 GB')

        download_sentinel2 = Transformation('download_sentinel2', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/download_sentinel2.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='4 GB')

        preprocess_atl03 = Transformation('preprocess_atl03', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/preprocess_atl03.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='8 GB')

        auto_label = Transformation('auto_label', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/auto_label.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='4 GB')

        train_model = Transformation('train_model', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/train_model.py'),
            is_stageable=True, container=seaice_gpu_container,
        ).add_pegasus_profile(memory='16 GB'
        ).add_condor_profile(request_gpus=1)

        classify_seaice = Transformation('classify_seaice', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/classify_seaice.py'),
            is_stageable=True, container=seaice_gpu_container,
        ).add_pegasus_profile(memory='8 GB'
        ).add_condor_profile(request_gpus=1)

        merge_classifications = Transformation('merge_classifications', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/merge_classifications.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='4 GB')

        calculate_freeboard = Transformation('calculate_freeboard', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/calculate_freeboard.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='8 GB')

        visualize_results = Transformation('visualize_results', site=exec_site_name,
            pfn=os.path.join(self.wf_dir, 'bin/visualize_results.py'),
            is_stageable=True, container=seaice_container,
        ).add_pegasus_profile(memory='4 GB')

        self.tc.add_containers(seaice_container, seaice_gpu_container)
        self.tc.add_transformations(
            download_atl03, download_sentinel2, preprocess_atl03, auto_label,
            train_model, classify_seaice, merge_classifications,
            calculate_freeboard, visualize_results,
        )

    # --- Replica Catalog ---
    def create_replica_catalog(self):
        self.rc = ReplicaCatalog()

        if self.test_mode:
            test_data_dir = os.path.join(self.wf_dir, 'test_data')
            atl03_path = os.path.join(test_data_dir, 'atl03_data.h5')
            labeled_path = os.path.join(test_data_dir, 'labeled_data.csv')

            if not os.path.exists(atl03_path):
                print(f'WARNING: Test data not found at {test_data_dir}. '
                      'Run generate_test_data.py first.')

            self.rc.add_replica('local', 'atl03_data.h5', 'file://' + atl03_path)
            self.rc.add_replica('local', 'labeled_data.csv', 'file://' + labeled_path)
            print(f'Registered test data from {test_data_dir}')

    # --- Create Workflow DAG ---
    def create_workflow(self):
        self.wf = Workflow(self.wf_name, infer_dependencies=True)

        # Output files
        atl03_data = File('atl03_data.h5')
        atl03_bbox = File('atl03_bbox.json')
        sentinel2_scenes = File('sentinel2_scenes.tar.gz')
        atl03_preprocessed = File('atl03_preprocessed.csv')
        labeled_data = File('labeled_data.csv')
        model_file = File('model.h5')
        training_metrics = File('training_metrics.json')
        classification_results = File('classification_results.csv')
        freeboard_results = File('freeboard_results.csv')
        classification_map = File('classification_map.png')
        freeboard_profile = File('freeboard_profile.png')
        summary_stats = File('summary_statistics.json')

        if self.test_mode:
            print('TEST MODE: Skipping download jobs, using synthetic test data')
        else:
            # Job 1: Download ATL03
            download_atl03_args = [
                '--region', self.region,
                '--start-date', self.start_date,
                '--end-date', self.end_date,
                '--output', atl03_data,
            ]
            if self.max_granules:
                download_atl03_args.extend(['--max-granules', str(self.max_granules)])

            download_atl03_job = (
                Job('download_atl03', _id='download_atl03', node_label='download_atl03')
                .add_args(*download_atl03_args)
                .add_outputs(atl03_data, stage_out=True, register_replica=False)
                .add_outputs(atl03_bbox, stage_out=False, register_replica=False)
            )
            if self.earthdata_token and self.earthdata_token != 'YOUR_TOKEN_HERE':
                download_atl03_job.add_env(EARTHDATA_TOKEN=self.earthdata_token)
            elif self.earthdata_username and self.earthdata_password:
                download_atl03_job.add_env(
                    EARTHDATA_USERNAME=self.earthdata_username,
                    EARTHDATA_PASSWORD=self.earthdata_password,
                )
            self.wf.add_jobs(download_atl03_job)

            # Job 2: Download Sentinel-2
            download_s2_args = [
                '--bbox-file', atl03_bbox,
                '--start-date', self.start_date,
                '--end-date', self.end_date,
                '--output', sentinel2_scenes,
            ]
            if self.max_scenes:
                download_s2_args.extend(['--max-scenes', str(self.max_scenes)])

            download_s2_job = (
                Job('download_sentinel2', _id='download_sentinel2', node_label='download_sentinel2')
                .add_args(*download_s2_args)
                .add_inputs(atl03_bbox)
                .add_outputs(sentinel2_scenes, stage_out=True, register_replica=False)
            )
            self.wf.add_jobs(download_s2_job)

        # Job 3: Preprocess ATL03
        preprocess_job = (
            Job('preprocess_atl03', _id='preprocess_atl03', node_label='preprocess_atl03')
            .add_args('--input', atl03_data, '--output', atl03_preprocessed)
            .add_inputs(atl03_data)
            .add_outputs(atl03_preprocessed, stage_out=True, register_replica=False)
        )
        self.wf.add_jobs(preprocess_job)

        if self.test_mode:
            print('TEST MODE: Skipping auto_label job, using pre-built labeled_data.csv')
        else:
            # Job 4: Auto-label
            auto_label_job = (
                Job('auto_label', _id='auto_label', node_label='auto_label')
                .add_args(
                    '--atl03-input', atl03_preprocessed,
                    '--sentinel2-input', sentinel2_scenes,
                    '--output', labeled_data,
                )
                .add_inputs(atl03_preprocessed, sentinel2_scenes)
                .add_outputs(labeled_data, stage_out=True, register_replica=False)
            )
            self.wf.add_jobs(auto_label_job)

        # Job 5: Train model
        train_job = (
            Job('train_model', _id='train_model', node_label='train_model')
            .add_args(
                '--input', labeled_data,
                '--model-output', model_file,
                '--metrics-output', training_metrics,
                '--model-type', self.model_type,
            )
            .add_inputs(labeled_data)
            .add_outputs(model_file, stage_out=True, register_replica=False)
            .add_outputs(training_metrics, stage_out=True, register_replica=False)
        )
        self.wf.add_jobs(train_job)

        # Job 6: Classify sea ice (fan-out when max_granules or test_mode)
        num_classify_jobs = None
        if self.test_mode:
            num_classify_jobs = 2
        elif self.max_granules is not None:
            num_classify_jobs = self.max_granules

        if num_classify_jobs is not None and num_classify_jobs > 1:
            per_granule_files = []
            for i in range(num_classify_jobs):
                granule_id_str = f'granule_{i:04d}'
                per_granule_file = File(f'classification_{granule_id_str}.csv')
                per_granule_files.append(per_granule_file)

                classify_job_i = (
                    Job('classify_seaice', _id=f'classify_seaice_{i}',
                        node_label=f'classify_seaice_{i}')
                    .add_args(
                        '--input', atl03_preprocessed,
                        '--model', model_file,
                        '--granule', granule_id_str,
                        '--output', per_granule_file,
                    )
                    .add_inputs(atl03_preprocessed, model_file)
                    .add_outputs(per_granule_file, stage_out=False, register_replica=False)
                )
                self.wf.add_jobs(classify_job_i)

            merge_args = ['--output', classification_results, '--inputs']
            merge_args.extend(per_granule_files)

            merge_job = (
                Job('merge_classifications', _id='merge_classifications',
                    node_label='merge_classifications')
                .add_args(*merge_args)
                .add_inputs(*per_granule_files)
                .add_outputs(classification_results, stage_out=True, register_replica=False)
            )
            self.wf.add_jobs(merge_job)
            print(f'Fan-out: {num_classify_jobs} parallel classify jobs + merge')
        else:
            classify_job = (
                Job('classify_seaice', _id='classify_seaice', node_label='classify_seaice')
                .add_args(
                    '--input', atl03_preprocessed,
                    '--model', model_file,
                    '--output', classification_results,
                )
                .add_inputs(atl03_preprocessed, model_file)
                .add_outputs(classification_results, stage_out=True, register_replica=False)
            )
            self.wf.add_jobs(classify_job)
            print('Single classify job (no fan-out)')

        # Job 7: Calculate freeboard
        freeboard_job = (
            Job('calculate_freeboard', _id='calculate_freeboard',
                node_label='calculate_freeboard')
            .add_args('--input', classification_results, '--output', freeboard_results)
            .add_inputs(classification_results)
            .add_outputs(freeboard_results, stage_out=True, register_replica=False)
        )
        self.wf.add_jobs(freeboard_job)

        # Job 8: Visualize results
        visualize_job = (
            Job('visualize_results', _id='visualize_results', node_label='visualize_results')
            .add_args(
                '--classification-input', classification_results,
                '--freeboard-input', freeboard_results,
                '--classification-map-output', classification_map,
                '--freeboard-profile-output', freeboard_profile,
                '--summary-output', summary_stats,
            )
            .add_inputs(classification_results, freeboard_results)
            .add_outputs(classification_map, stage_out=True, register_replica=False)
            .add_outputs(freeboard_profile, stage_out=True, register_replica=False)
            .add_outputs(summary_stats, stage_out=True, register_replica=False)
        )
        self.wf.add_jobs(visualize_job)


# --- Build and generate the workflow ---
dagfile = 'workflow.yml'

workflow = SeaIceWorkflow(
    region=REGION,
    start_date=START_DATE,
    end_date=END_DATE,
    model_type=MODEL_TYPE,
    max_granules=MAX_GRANULES,
    max_scenes=MAX_SCENES,
    test_mode=TEST_MODE,
    dagfile=dagfile,
)

print('Creating execution sites...')
workflow.create_sites_catalog('condorpool')

print('Creating workflow properties...')
workflow.create_pegasus_properties()

print('Creating transformation catalog...')
workflow.create_transformation_catalog('condorpool')

print('Creating replica catalog...')
workflow.create_replica_catalog()

print('Creating sea ice workflow DAG...')
workflow.create_workflow()

workflow.write()
print('\nSea Ice Workflow has been generated!')

## View the Generated Workflow DAG

Before submitting, we can visualize the workflow DAG using `pegasus-graphviz`. The graph shows all jobs as nodes and data dependencies as edges. When `MAX_GRANULES` is set, you should see parallel classify jobs fanning out from `train_model` and merging back before `calculate_freeboard`.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow_dag.png

In [ ]:
from IPython.display import Image
Image(filename='workflow_dag.png')

## 2. Plan and Submit the Workflow

We will now plan and submit the workflow for execution. By default we are running jobs on site **condorpool** i.e. the selected ACCESS resource.

In [ ]:
workflow.plan_submit()

After the workflow has been successfully planned and submitted, you can use the Python `Workflow` object to monitor the status of the workflow. It shows in detail the counts of jobs of each status and whether a job is idle or running.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 3. Statistics

Depending on whether the workflow finished successfully or not, you have options on what to do next. If the workflow failed you can use `workflow.analyze()` to get help finding out what went wrong. If the workflow finished successfully, we can pull out some statistics from the provenance database:

In [ ]:
workflow.statistics()

## 4. Examining the Results

Once the workflow has finished, we can look at the output directory for our results:

```
output/
├── atl03_data.h5                  # Raw ATL03 photon data (HDF5)
├── sentinel2_scenes.tar.gz        # Sentinel-2 band imagery
├── atl03_preprocessed.csv         # 2m segment features
├── labeled_data.csv               # Auto-labeled training data
├── model.h5                       # Trained LSTM/MLP classifier weights
├── training_metrics.json          # Training loss/accuracy history
├── classification_results.csv     # Per-segment ice type predictions
├── freeboard_results.csv          # Per-segment freeboard values
├── classification_map.png         # Classification visualization
├── freeboard_profile.png          # Freeboard visualization
└── summary_statistics.json        # Aggregate statistics
```

In [ ]:
!ls -ltR output/

### Classification Map

The classification visualization shows:
- **(a)** ATL03 elevation vs along-track longitude colored by ice type (thick ice = blue, thin ice = green, open water = orange)
- **(b)** Geographic classification map (lat vs lon)
- **(c)** Confusion matrix with per-class percentages (when ground-truth labels are available)

In [ ]:
from IPython.display import Image, display
import glob

cls_maps = glob.glob('output/classification_map.png')
for png in cls_maps:
    print(f'\n{png}')
    display(Image(filename=png))

### Freeboard Profile

The freeboard visualization shows:
- **(a)** Elevation profile with sea surface detection line
- **(b)** Freeboard vs along-track longitude
- **(c)** Freeboard value distribution histogram
- **(d)** Point density along track

In [ ]:
fb_profiles = glob.glob('output/freeboard_profile.png')
for png in fb_profiles:
    print(f'\n{png}')
    display(Image(filename=png))

### Summary Statistics

The summary statistics include classification counts, percentages, prediction confidence, freeboard statistics per ice type, and geographic extent.

In [ ]:
import json

summary_files = glob.glob('output/summary_statistics.json')
for summary_file in summary_files:
    with open(summary_file, 'r') as f:
        stats = json.load(f)

    print(f'Total segments: {stats["total_segments"]:,}')
    if stats.get('track_length_km', 0) > 0:
        print(f'Track length: {stats["track_length_km"]:.1f} km')

    print(f'\nClassification:')
    for cls_name, cls_stats in stats['classification'].items():
        if isinstance(cls_stats, dict):
            line = f'  {cls_name}: {cls_stats["count"]:,} ({cls_stats["percentage"]:.1f}%)'
            if 'accuracy' in cls_stats:
                line += f'  [accuracy: {cls_stats["accuracy"]:.2%}]'
            print(line)

    if 'all_ice' in stats.get('freeboard', {}):
        fb = stats['freeboard']['all_ice']
        print(f'\nFreeboard (all ice):')
        print(f'  Mean: {fb["mean"]:.3f} m')
        print(f'  Median: {fb["median"]:.3f} m')
        print(f'  Std: {fb["std"]:.3f} m')

    for ice_type in ['thick_ice', 'thin_ice']:
        if ice_type in stats.get('freeboard', {}):
            fb = stats['freeboard'][ice_type]
            print(f'  {ice_type}: mean={fb["mean"]:.3f} m, '
                  f'median={fb["median"]:.3f} m, n={fb["count"]:,}')